<a href="https://colab.research.google.com/github/san-258/1-branch/blob/main/SwingtradingSnippet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Your swing trading strategy uses several technical indicators and conditions to generate buy signals. Here's a breakdown:

Indicators Used:

Moving Averages (SMA): Simple Moving Averages of 10, 50, and 200 periods are calculated on the closing price. The slope of the 10-period SMA is also considered.
Volume SMAs: Simple Moving Averages of 10 and 50 periods are calculated on the trading volume.
Relative Strength Index (RSI): The 14-period RSI is used to identify overbought conditions.
Volatility: Calculated as the rolling standard deviation of daily returns over a 20-day window.
Entry Conditions (All must be met for a 'Swing Signal'):

SMA Crossover: The closing price must be above both the 50-period and 10-period Simple Moving Averages.
Volume Confirmation: Either:
A significant single-day increase in volume (greater than 1.5 times the 10-period Volume SMA).
A streak of 3 consecutive days where volume is higher than 1.5 times the 50-period Volume SMA.
RSI Not Overbought: The 14-period RSI must be below 70.
Price Breakout: The closing price must be above the previous day's high.
Above Long-Term SMA: The closing price must be above the 200-period Simple Moving Average.
Positive Short-Term Trend: The slope of the 10-period Simple Moving Average must be positive.
Exit Conditions (First condition met triggers an exit):

Time-Based Exit: Exit after 10 trading days from the entry date.
Stop-Loss: Exit if the price drops 8% below the entry price.
Take-Profit: Exit if the price rises 12% above the entry price.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf # Assuming yfinance is used for data acquisition

def calculate_rsi(data, window):
    """Calculates the Relative Strength Index (RSI) for a given data series."""
    diff = data.diff(1).dropna()
    gain = diff.mask(diff < 0, 0)
    loss = diff.mask(diff > 0, 0).abs()

    avg_gain = gain.ewm(com=window - 1, adjust=False).mean()
    avg_loss = loss.ewm(com=window - 1, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_higher_volume_streak(volume, volume_sma, threshold=1.5, window=3):
    """Identifies streaks of higher volume relative to its SMA."""
    volume_ratio = volume / volume_sma
    # Identify days where volume is above the threshold
    high_volume_days = volume_ratio > threshold
    # Use a rolling sum to find streaks of high volume days
    # A rolling sum equal to the window indicates a streak of that length
    volume_streak = high_volume_days.rolling(window=window).sum() == window
    return volume_streak


def apply_swing_trading_strategy(df, ticker, single_day_volume_multiplier=1.5, volume_streak_multiplier=1.5, volume_streak_window=3, sma10_slope_periods=2, rsi_overbought_threshold=70):
    """
     Applies the final refined swing trading strategy to a stock's historical data.

    Args:
        df (pd.DataFrame): DataFrame containing historical stock data.
        ticker (str): Stock ticker symbol.
        single_day_volume_multiplier (float): Multiplier for the single-day volume surge relative to Volume_SMA_10.
        volume_streak_multiplier (float): Multiplier for defining volume streak relative to Volume_SMA_50.
        volume_streak_window (int): Number of periods for the volume streak.
        sma10_slope_periods (int): Number of periods to calculate SMA_10 slope.
        rsi_overbought_threshold (int): RSI value considered overbought.


    Returns:
        pd.DataFrame: DataFrame with added technical indicators and 'Swing_Signal'.
    """
    # Flatten the MultiIndex columns if they exist
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(col).strip() for col in df.columns.values]

    # Construct expected column names after flattening
    close_col = f'Close_{ticker}' if f'Close_{ticker}' in df.columns else 'Close'
    high_col = f'High_{ticker}' if f'High_{ticker}' in df.columns else 'High'
    volume_col = f'Volume_{ticker}' if f'Volume_{ticker}' in df.columns else 'Volume'

    # Check if required columns exist after flattening and name adjustment
    required_cols_check = [close_col, high_col, volume_col]
    if not all(col in df.columns for col in required_cols_check):
        print(f"Skipping strategy application for {ticker} due to missing required columns: {required_cols_check}.")
        return df # Return original df if columns are missing


    # Ensure consistent column access
    close_price = df[close_col]
    high_price = df[high_col]
    volume = df[volume_col]

    # Calculate Daily Return
    df['Daily_Return'] = close_price.pct_change()

    # Calculate SMAs
    df['SMA_10'] = close_price.rolling(window=10).mean()
    df['SMA_50'] = close_price.rolling(window=50).mean()
    df['SMA_200'] = close_price.rolling(window=200).mean()

    # Calculate SMA_10 slope
    df['SMA_10_Slope'] = (df['SMA_10'] - df['SMA_10'].shift(sma10_slope_periods)) / sma10_slope_periods

    # Calculate Volume SMAs (including 10-day and 50-day for combined volume condition)
    df['Volume_SMA_10'] = volume.rolling(window=10).mean()
    df['Volume_SMA_50'] = volume.rolling(window=50).mean()


    # Calculate Volume Change
    df['Volume_Change'] = volume.pct_change()

    # Calculate RSI
    df['RSI'] = calculate_rsi(close_price, window=14)

    # Calculate Volatility
    df['Volatility'] = df['Daily_Return'].rolling(window=20).std()

    # Define final swing trading conditions
    # Condition 1: Price crossing above SMA_50 and SMA_10
    condition_sma_cross = (df[close_col] > df['SMA_50']) & (df[close_col] > df['SMA_10'])

    # Condition 2a: Significant single-day increase in volume (relative to Volume_SMA_10)
    condition_single_day_volume_surge = df[volume_col] > df['Volume_SMA_10'] * single_day_volume_multiplier

    # Condition 2b: Higher Volume Streak (relative to Volume_SMA_50)
    condition_volume_streak = calculate_higher_volume_streak(volume, df['Volume_SMA_50'], threshold=volume_streak_multiplier, window=volume_streak_window)

    # Combine volume conditions using logical OR
    condition_volume = condition_single_day_volume_surge | condition_volume_streak

    # Condition 3: RSI indicating not overbought
    condition_rsi_not_overbought = df['RSI'] < rsi_overbought_threshold

    # Condition 4: Price breakout (closing above previous day's high)
    condition_breakout = df[close_col] > df[high_col].shift(1)

    # Condition 5: Close price above SMA_200
    condition_above_sma200 = (df[close_col] > df['SMA_200'])

    # Condition 6: SMA_10 slope is positive
    condition_sma10_slope = df['SMA_10_Slope'] > 0


    # Combine all conditions using logical operators
    df['Swing_Signal'] = condition_sma_cross & condition_volume & condition_rsi_not_overbought & condition_breakout & condition_above_sma200 & condition_sma10_slope

    return df

# --- Backtesting Logic ---

# Define backtesting parameters (based on Iteration 12 results)
hold_period = 10 # Final 10 trading days hold period
stop_loss_pct = 0.08 # Final 8% stop loss
take_profit_pct = 0.12 # Final 12% take profit

# Assuming 'stock_data' dictionary contains the historical data for each ticker
# If starting fresh, you would first need to acquire data using yfinance or another method
# Example data acquisition (if starting fresh):
sp500_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA'] # Define your list of tickers
start_date = '2022-01-01'
end_date = '2024-07-31'
stock_data = {}
for ticker in sp500_tickers:
    try:
        data = yf.download(ticker, start=start_date, end=end_date)
        if not data.empty:
            stock_data[ticker] = data
    except Exception as e:
        print(f"Error downloading data for {ticker}: {e}")


trade_results = []

for ticker, df in stock_data.items():
    # Apply the swing trading strategy to get signals
    processed_df = apply_swing_trading_strategy(df.copy(),
                                                ticker,
                                                single_day_volume_multiplier=1.5,
                                                volume_streak_multiplier=1.5,
                                                volume_streak_window=3,
                                                sma10_slope_periods=2,
                                                rsi_overbought_threshold=70)

    # Ensure the DataFrame is sorted by date
    processed_df = processed_df.sort_index()

    # Construct expected column names after flattening
    close_col = f'Close_{ticker}' if f'Close_{ticker}' in processed_df.columns else 'Close'
    high_col = f'High_{ticker}' if f'High_{ticker}' in processed_df.columns else 'High'
    volume_col = f'Volume_{ticker}' if f'Volume_{ticker}' in processed_df.columns else 'Volume'


    # Skip if essential columns are missing
    if not all(col in processed_df.columns for col in [close_col, high_col, volume_col, 'Swing_Signal']):
        print(f"Skipping backtesting for {ticker} due to missing required columns after strategy application.")
        continue

    # Identify entry points (Swing_Signal is True)
    entry_dates = processed_df[processed_df['Swing_Signal']].index

    for entry_date in entry_dates:
        try:
            entry_price = processed_df.loc[entry_date, close_col]
            entry_index = processed_df.index.get_loc(entry_date)

            # Define potential exit range
            potential_exit_end_index = min(entry_index + hold_period + 1, len(processed_df)) # +1 to include the end date
            potential_exit_dates = processed_df.index[entry_index + 1 : potential_exit_end_index]


            if potential_exit_dates.empty:
                 continue # Skip if no potential exit dates within hold period

            # Define exit date (time-based)
            exit_date_time = potential_exit_dates[-1]

            # Define exit date (price-based stop-loss/take-profit) using final percentages
            stop_loss_price = entry_price * (1 - stop_loss_pct)
            take_profit_price = entry_price * (1 + take_profit_pct)

            # Find the first date within the potential exit range where stop-loss or take-profit is hit
            price_exit_df = processed_df.loc[potential_exit_dates].copy()

            # Find the first date the close price is below stop loss
            stop_loss_date = price_exit_df[price_exit_df[close_col] <= stop_loss_price].index.min()

            # Find the first date the close price is above take profit
            take_profit_date = price_exit_df[price_exit_df[close_col] >= take_profit_price].index.min()

            # Determine the actual exit date and price based on the earliest condition met
            exit_date = exit_date_time # Default to time-based exit
            exit_price = processed_df.loc[exit_date, close_col]

            if pd.notna(stop_loss_date):
                 exit_date = min(exit_date, stop_loss_date)
                 exit_price = processed_df.loc[exit_date, close_col]

            if pd.notna(take_profit_date):
                 exit_date = min(exit_date, take_profit_date)
                 exit_price = processed_df.loc[exit_date, close_col]


            # Calculate return
            trade_return = (exit_price - entry_price) / entry_price


            # Store the trade result
            trade_results.append({
                'Ticker': ticker,
                'Entry_Date': entry_date,
                'Exit_Date': exit_date,
                'Entry_Price': entry_price,
                'Exit_Price': exit_price,
                'Return': trade_return
            })
        except KeyError:
            print(f"Skipping trade for {ticker} on {entry_date} due to missing data.")
            continue
        except IndexError:
             print(f"Skipping trade for {ticker} on {entry_date} due to insufficient data after entry.")
             continue


# Convert results to a DataFrame
trade_df = pd.DataFrame(trade_results)

# Calculate summary statistics
total_trades = len(trade_df)
profitable_trades = len(trade_df[trade_df['Return'] > 0])
win_rate = profitable_trades / total_trades if total_trades > 0 else 0
average_return = trade_df['Return'].mean() if total_trades > 0 else 0
cumulative_return = (1 + trade_df['Return']).prod() - 1 if total_trades > 0 else 0

# Print summary statistics
print("\n--- Final Strategy Performance Summary ---")
print(f"Total Trades: {total_trades}")
print(f"Profitable Trades: {profitable_trades}")
print(f"Win Rate: {win_rate:.2%}")
print(f"Average Return per Trade: {average_return:.2%}")
print(f"Cumulative Return: {cumulative_return:.2%}")

# Display the first few trade results
print("\n--- Sample Trade Results ---")
display(trade_df.head())

/tmp/ipython-input-3372549215.py:139: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3372549215.py:139: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3372549215.py:139: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3372549215.py:139: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-inp


--- Final Strategy Performance Summary ---
Total Trades: 21
Profitable Trades: 18
Win Rate: 85.71%
Average Return per Trade: 3.42%
Cumulative Return: 100.24%

--- Sample Trade Results ---


,Ticker,Entry_Date,Exit_Date,Entry_Price,Exit_Price,Return
0,AAPL,2023-05-05,2023-05-19,171.612045,173.423645,0.010556
1,AAPL,2023-12-05,2023-12-19,192.013855,195.508270,0.018199
2,AAPL,2024-05-03,2024-05-17,182.279129,188.986160,0.036795
3,AAPL,2024-05-31,2024-06-14,191.355103,211.500870,0.105279
4,MSFT,2023-03-15,2023-03-29,260.805634,275.612488,0.056774
